# Supervised learning testing file (before adding to pipeline script)

In [1]:
from met_cleaning import clean_met_data
from sklearn.model_selection import train_test_split
# from sklearn.metrics import confusion_matrix
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from imblearn.over_sampling import SMOTE


In [2]:
df = clean_met_data()

/Users/mollystark/Desktop/michigan/milestone 2/SIADS-Milestone-II/src/notebooks/met_cleaning.py:20: DtypeWarning: Columns (5,7,10,11,12,13,14,34,35,36,37,38,39,40,41,42,43,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  met = pd.read_csv('https://media.githubusercontent.com/media/metmuseum'


In [3]:
X = df.drop('is_significant', axis=1)
y = df['is_significant']

In [4]:
numeric_features = ['access_year', 'extracted_date']
numeric_transformer = Pipeline(steps=[("imputer",
                                       SimpleImputer(strategy="constant", fill_value=0)),])

In [5]:
categorical_features = ['object_name','culture','portfolio', #'title','department',
                        'artist_display_name','artist_nationality','mapped_country','medium', 'artist_gender']
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy="constant", 
                                                                    fill_value="missing")),
                                      ('onehot', OneHotEncoder(handle_unknown="ignore"))])

In [6]:
tag_features = ['tags']
tag_transformer = Pipeline(steps=[('tfidf', TfidfVectorizer())])

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        # ("tags", tag_transformer, tag_features)
    ]
)

In [8]:
pipe = Pipeline(steps=[#('preprocessor', preprocessor),
                       ('classifier', BalancedRandomForestClassifier())
                       ])
smote = SMOTE(random_state=42)

In [9]:
X = preprocessor.fit_transform(X)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8,random_state=42)

In [11]:
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [12]:
# pipe.fit(X_train_resampled, y_train_resampled)

In [13]:
clf = BalancedRandomForestClassifier(random_state=42)
clf.fit(X_train_resampled, y_train_resampled)

BalancedRandomForestClassifier(random_state=42)

In [14]:
# metrics = {
#     "train_data":{
#         "score": pipe.score(X_train, y_train),
#         "mae": mean_absolute_error(y_train, pipe.predict(X_train))
#     },
#     "test_data":{
#         "score": pipe.score(X_test, y_test),
#         "mae": mean_absolute_error(y_test, pipe.predict(X_test))
#     }
# }

metrics = {
    "train_data":{
        "score": clf.score(X_train, y_train),
        "mae": mean_absolute_error(y_train, clf.predict(X_train))
    },
    "test_data":{
        "score": clf.score(X_test, y_test),
        "mae": mean_absolute_error(y_test, clf.predict(X_test))
    }
}

print(metrics)


{'train_data': {'score': 0.9989792867379448, 'mae': 0.0010207132620552422}, 'test_data': {'score': 0.9804853530601987, 'mae': 0.01951464693980127}}
